# 07 · Generation and attention in small examples

Ting Tan · EEEA · Oct 2026

Compute a next-token probability, generate a short sequence and read a causal attention matrix.

0–10 min: predict; 10–25: worked example; 25–45: guided run; 45–65: one change; 65–80: interpret; 80–90: explain and submit.

## A1 · Predict before running

The word a is followed by book 3 times and note 4 times in the English corpus. What are the two empirical next-token probabilities?

## A2 · Understand the idea

The local corpus contains seven short fictional sentences per language. A token is a whitespace-separated word in this experiment. A bigram model estimates the next word using the current word: P(next|current)=count(current,next)/sum_next count(current,next). Generation repeatedly samples a next token and appends it until END or the 12-new-token budget. The model represents local word-pair counts. Its generated sentences provide examples for discussing repetition, coherence and factual support.

Temperature transforms positive probabilities as p_T(i)=p(i)^(1/T)/sum_j p(j)^(1/T). A smaller positive T favours the more probable token. The attention calculation is a separate fixed numerical example: scores=QKᵀ/sqrt(d), mask future positions, apply row-wise softmax, then multiply by V. Each row sums to 1 and future-position weights equal 0. This demonstrates a Transformer building block; the word generator uses the bigram probabilities.

Run the setup cell once. It locates the course folder and imports the supplied tools.

In [ ]:
from pathlib import Path
import sys
candidates = [q for p in [Path.cwd(), *Path.cwd().parents] for q in [p, p / "ML_Labs_2026"]]
ROOT = next(p for p in candidates if (p / "ml_labs.py").exists())
sys.path.insert(0, str(ROOT))
from ml_labs import run_lab, show, split_data, read_rows
from feedback import check_answer
import numpy as np
LANG = "en"
print("Course folder:", ROOT.name)
print("Python:", sys.version.split()[0])

## A3 · Worked example

In [ ]:
import json
from models import Bigram, causal_attention
sentences = json.loads((ROOT / "data" / "corpus.json").read_text(encoding="utf-8"))[LANG]
model = Bigram(sentences)
context = "a" if LANG == "en" else "un"
tokens, probabilities = model.distribution(context)
print(list(zip(tokens, probabilities)))
Q = np.array([[1.,0.], [0.,1.], [1.,1.]])
V = np.array([[10.,0.], [0.,10.], [10.,10.]])
attention_output, weights = causal_attention(Q, Q, V)
print(weights)
print("row sums:", weights.sum(axis=1))

The probabilities are 3/7 and 4/7. The first attention row is [1,0,0], so its output equals V[0]=[10,0]. The second row is approximately [0.33024,0.66976,0]. At T=0.5, the two probabilities become 9/25 and 16/25.

## B1 · Write P(note|a) from the English corpus.

None marks a response you will complete. This cell is independent of the model experiment.

In [ ]:
student_probability = None
check_answer(student_probability, 4 / 7, LANG)

<details><summary>Hint 1</summary>

Use the count of this pair as numerator.

</details>

<details><summary>Hint 2</summary>

Divide by the sum of counts of all continuations of the context.

</details>

## B2 · Run the baseline and read the axes

Explain why changing temperature changes sampling while leaving learned word-pair counts fixed. Read one attention row as a weighted average of V. Compare the output of classification with the output of generation.

In [ ]:
baseline = run_lab(7, LANG, ROOT / "outputs" / LANG / "lab07_baseline")
show(baseline)

## B3 · Change one thing

Compare temperatures 1 and 0.5 with the same corpus and sampling seeds. Compare the probability table first, then the generated strings. Explain which observations concern generation and which concern the separate attention example.

In [ ]:
temperature = 0.5
changed = run_lab(7, LANG, ROOT / "outputs" / LANG / "lab07_changed", temperature=temperature)
show(changed)

## C · Optional exploration: choose one additional input or parameter value. Predict its effect, run the experiment and justify your observation.

## Explain your result

Submit the 3/7 and 4/7 derivation, the temperature calculation, an attention-row explanation and one generated-text critique.

My prediction:

My observation:

My explanation:

One question to discuss: